In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_spark = spark.table("feature_er_visit_model")
fact_visits = spark.table("fact_er_visit")

In [0]:
# Complaint grouping
df_spark = (
    df_spark
    .withColumn(
        "complaint_group",
        F.when(F.col("chief_complaint_raw").isin("Cough", "Fever", "Shortness Of Breath"), "Respiratory Infection")
         .when(F.col("chief_complaint_raw").isin("Chest Pain", "Abdominal Pain", "Back Pain", "Headache"), "Pain")
         .when(F.col("chief_complaint_raw").isin("Fall", "Injury", "Laceration"), "Injury Trauma")
         .when(F.col("chief_complaint_raw").isin("Dizziness", "Weakness"), "Neuro General")
         .when(F.col("chief_complaint_raw").isin("Urinary Symptoms", "Nausea Vomiting"), "Medical Other")
         .otherwise("Other")
    )
)

# Create date and hour keys
df_spark = (
    df_spark
    .withColumn("arrival_date", F.to_date("arrival_ts"))
    .withColumn("arrival_hour_ts", F.date_trunc("hour", F.col("arrival_ts")))
)

# Hourly ER volume
hourly_volume = (
    df_spark
    .groupBy("arrival_hour_ts")
    .agg(F.count("*").alias("er_arrivals_hour"))
)

# Daily ER volume
daily_volume = (
    df_spark
    .groupBy("arrival_date")
    .agg(F.count("*").alias("er_arrivals_day"))
)

# Provider hourly workload
provider_hourly = (
    df_spark
    .groupBy("provider_id", "arrival_hour_ts")
    .agg(F.count("*").alias("provider_hourly_load"))
)

# Join engineered features back
df_spark = (
    df_spark
    .join(hourly_volume, on="arrival_hour_ts", how="left")
    .join(daily_volume, on="arrival_date", how="left")
    .join(provider_hourly, on=["provider_id", "arrival_hour_ts"], how="left")
)

In [0]:
delay_df = (
    df_spark
    .filter(F.col("wait_to_provider_min").isNotNull())
    .filter(F.col("visit_status") == "Completed")
    .select(
        "visit_id",
        "delay_flag",
        "arrival_mode",
        "acuity_cd",
        "chief_complaint_raw",
        "complaint_group",
        "arrival_hour",
        "arrival_day_of_week",
        "arrival_month",
        "is_weekend",
        "is_peak_hour",
        "gender_cd",
        "age_band_raw",
        "insurance_plan",
        "chronic_cnt",
        "risk_ind",
        "provider_role_raw",
        "shift_code",
        "er_arrivals_hour",
        "er_arrivals_day",
        "provider_hourly_load"
    )
)

delay_pd = delay_df.toPandas()
print(delay_pd.shape)
delay_pd.head()

(18249, 21)


,visit_id,delay_flag,arrival_mode,acuity_cd,chief_complaint_raw,complaint_group,arrival_hour,arrival_day_of_week,arrival_month,is_weekend,is_peak_hour,gender_cd,age_band_raw,insurance_plan,chronic_cnt,risk_ind,provider_role_raw,shift_code,er_arrivals_hour,er_arrivals_day,provider_hourly_load
0,V0000004,1,Walk In,4,Other,Other,4,Mon,1,0,0,M,50_64,Commercial,1,N,Attending,NGT,1,43,1.0
1,V0000010,1,Walk In,5,Cough,Respiratory Infection,7,Mon,1,0,0,F,00_17,Commercial,1,N,Attending,DAY,1,43,1.0
2,V0000011,0,Ambulance,3,Fever,Respiratory Infection,8,Mon,1,0,0,M,50_64,Commercial,1,N,Pa,EVE,2,43,1.0
3,V0000012,1,Walk In,3,Fall,Injury Trauma,8,Mon,1,0,0,M,18_34,Commercial,1,N,Resident,DAY,2,43,1.0
4,V0000015,1,Transfer,4,Dizziness,Neuro General,9,Mon,1,0,0,F,65_PLUS,Medicare,0,Y,Pa,DAY,2,43,1.0


In [0]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

target_col = "delay_flag"

X = delay_pd.drop(columns=[target_col, "visit_id"])
y = delay_pd[target_col]

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (14599, 19)
Test shape: (3650, 19)


In [0]:
models = {
    "logistic_regression": LogisticRegression(max_iter=1500, class_weight="balanced"),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=8,
        random_state=42,
        class_weight="balanced"
    ),
    "gradient_boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}

results = {}
fitted_models = {}

for model_name, model in models.items():
    clf = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_prob)

    results[model_name] = auc
    fitted_models[model_name] = clf

    print("\nModel:", model_name)
    print("AUC:", round(auc, 4))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("Classification Report:")
    print(classification_report(y_test, y_pred))


Model: logistic_regression
AUC: 0.8296
Confusion Matrix:
[[1573  263]
 [ 712 1102]]
Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.86      0.76      1836
           1       0.81      0.61      0.69      1814

    accuracy                           0.73      3650
   macro avg       0.75      0.73      0.73      3650
weighted avg       0.75      0.73      0.73      3650


Model: random_forest
AUC: 0.8225
Confusion Matrix:
[[1624  212]
 [ 761 1053]]
Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.88      0.77      1836
           1       0.83      0.58      0.68      1814

    accuracy                           0.73      3650
   macro avg       0.76      0.73      0.73      3650
weighted avg       0.76      0.73      0.73      3650


Model: gradient_boosting
AUC: 0.8334
Confusion Matrix:
[[1531  305]
 [ 641 1173]]
Classification Report:
              precision    rec

In [0]:
# Final model selection without RF tuning
final_model = fitted_models["logistic_regression"]
final_model_name = "logistic_regression"
final_auc = results["logistic_regression"]

print("Final selected model:", final_model_name)
print("Final selected AUC:", round(final_auc, 4))

Final selected model: logistic_regression
Final selected AUC: 0.8296


In [0]:
final_prob = final_model.predict_proba(X_test)[:, 1]

for threshold in [0.30, 0.40, 0.50, 0.60]:
    pred_t = (final_prob >= threshold).astype(int)
    print("\nThreshold:", threshold)
    print(confusion_matrix(y_test, pred_t))
    print(classification_report(y_test, pred_t))


Threshold: 0.3
[[ 816 1020]
 [  92 1722]]
              precision    recall  f1-score   support

           0       0.90      0.44      0.59      1836
           1       0.63      0.95      0.76      1814

    accuracy                           0.70      3650
   macro avg       0.76      0.70      0.68      3650
weighted avg       0.76      0.70      0.67      3650


Threshold: 0.4
[[1173  663]
 [ 330 1484]]
              precision    recall  f1-score   support

           0       0.78      0.64      0.70      1836
           1       0.69      0.82      0.75      1814

    accuracy                           0.73      3650
   macro avg       0.74      0.73      0.73      3650
weighted avg       0.74      0.73      0.73      3650


Threshold: 0.5
[[1573  263]
 [ 712 1102]]
              precision    recall  f1-score   support

           0       0.69      0.86      0.76      1836
           1       0.81      0.61      0.69      1814

    accuracy                           0.73      3650

In [0]:
import pandas as pd

model = final_model.named_steps["model"]
preprocessor = final_model.named_steps["preprocessor"]

feature_names = preprocessor.get_feature_names_out()
coefficients = model.coef_[0]

importance_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

print("Top features increasing delay risk")
print(importance_df.sort_values("coefficient", ascending=False).head(15))

print("\nTop features reducing delay risk")
print(importance_df.sort_values("coefficient", ascending=True).head(15))

Top features increasing delay risk
                                       feature  coefficient
0                               num__acuity_cd     1.784314
68                cat__insurance_plan_Self Pay     1.544923
29  cat__chief_complaint_raw_Medication Refill     0.715115
21         cat__chief_complaint_raw_Chest_pain     0.297964
22              cat__chief_complaint_raw_Cough     0.239963
4                            num__is_peak_hour     0.231173
30  cat__chief_complaint_raw_Medication_refill     0.181968
13     cat__chief_complaint_raw_Abdominal_pain     0.174134
64                   cat__insurance_plan_Mcare     0.163183
14  cat__chief_complaint_raw_Allergic Reaction     0.152072
16          cat__chief_complaint_raw_Back Pain     0.146203
50                cat__arrival_day_of_week_Tue     0.125424
23          cat__chief_complaint_raw_Dizziness     0.118937
18  cat__chief_complaint_raw_Behavioral Health     0.103816
26           cat__chief_complaint_raw_Headache     0.073992

Top 

In [0]:
threshold = 0.4

scoring_df = (
    df_spark
    .select(
        "visit_id",
        "arrival_mode",
        "acuity_cd",
        "chief_complaint_raw",
        "complaint_group",
        "arrival_hour",
        "arrival_day_of_week",
        "arrival_month",
        "is_weekend",
        "is_peak_hour",
        "gender_cd",
        "age_band_raw",
        "insurance_plan",
        "chronic_cnt",
        "risk_ind",
        "provider_role_raw",
        "shift_code",
        "er_arrivals_hour",
        "er_arrivals_day",
        "provider_hourly_load"
    )
    .toPandas()
)

visit_ids = scoring_df["visit_id"].copy()
features = scoring_df.drop(columns=["visit_id"])

probs = final_model.predict_proba(features)[:, 1]

delay_scores_pd = pd.DataFrame({
    "visit_id": visit_ids,
    "pred_delay_risk_score": probs,
    "pred_delay_flag": (probs >= threshold).astype(int)
})

In [0]:
delay_scores_spark = spark.createDataFrame(delay_scores_pd)

delay_scores_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ml_delay_scores")

print("Saved ml_delay_scores")
spark.sql("SELECT COUNT(*) FROM ml_delay_scores").show()

Saved ml_delay_scores
+--------+
|COUNT(*)|
+--------+
|   20966|
+--------+



In [0]:
ml_delay_scores = spark.table("ml_delay_scores")

fact_er_visit_scored = (
    spark.table("fact_er_visit")
    .join(ml_delay_scores, on="visit_id", how="left")
)

fact_er_visit_scored.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fact_er_visit_scored")

print("Saved fact_er_visit_scored")
spark.sql("SELECT COUNT(*) FROM fact_er_visit_scored").show()

Saved fact_er_visit_scored
+--------+
|COUNT(*)|
+--------+
|   20966|
+--------+



In [0]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    AVG(pred_delay_risk_score) AS avg_risk,
    AVG(pred_delay_flag) AS predicted_delay_rate
FROM fact_er_visit_scored
""").show()

+----------+------------------+--------------------+
|total_rows|          avg_risk|predicted_delay_rate|
+----------+------------------+--------------------+
|     20966|0.5011740128853652|  0.6030716397977678|
+----------+------------------+--------------------+

